# Notebook 2: the Ca²⁺ wave around a wound (one dataset)

This notebook asks, for one wound movie: **does the Ca²⁺ response spread outward from the wound as a wave, how
far does it reach, and how fast does it travel?** The paper applies exactly this method to every wound dataset.

**Quick start (example data).** Run notebook 1 first (it writes the wound tracking), then run all cells here (in
Jupyter: *Run → Run All Cells*). The notebook is set up for the example dataset ctrl_0820 (control). Expected
result: an outward wave reaching 200 µm, speed 2.43 µm/s (95 % CI 1.39-4.09) (section 8).

**Your own data.** Follow the checklist in section 1.

**The method in short**
1. **Wound circle in every frame**, from the tracking of notebook 1.
2. **Sheet pixels and sectors.** The pixels reached by the sheet of responding cells that grows out from the
   wound edge are found (cells that flash on their own are left out; `ca_wave/sheet_wave.py`, the same code as
   the uncaging notebook). The tissue around the wound is cut into 11.25° wedges × 25-µm rings, measured from the
   wound edge in every frame, and the mean Ca²⁺ intensity of each sector's sheet pixels is taken in every frame.
3. **Normalization:** `(I - min)/min` per sector.
4. **Response and arrival time.** A sector responds if its trace rises clearly above its own noise; its arrival
   time is when it first reaches half of its peak.
5. **Reach and outward-order test.** How far the responses extend without a gap, how far the immediate response
   (within 21 s) extends, and whether the responses arrive later farther from the wound (permutation test).
6. **Speed**, only if step 5 shows an outward wave: a robust straight-line (Theil-Sen) fit of arrival time
   against distance.

**Contents**
1. Settings to edit (with the checklist for your own data)
2. Method settings (the values used in the paper)
3. Step 1: the wound circle in every frame
4. Steps 2-6: measurement and wave test
5. Saving and plotting helpers (tables and figures written)
6. Run
7. Summary
8. Result for the example
9. How to read the results, and troubleshooting

Appendix: settings used for the paper's datasets.

**Output** (in `results/wound_wave_analysis/<dataset>/`): tables of every sector and ring, a one-row summary
(`wave_summary.csv`), and six figures.

Paths: run this notebook from the `notebooks/` folder. Data and results folders default to `data/` and `results/` in the repository and can be changed with the environment variables listed in README.md.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge
from scipy.stats import theilslopes, kendalltau
from skimage.filters import gaussian
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

import sys
sys.path.insert(0, str(Path.cwd().parent))   # the repository root: the shared ca_wave package
from ca_wave.sheet_wave import SHEET_SETTINGS, sheet_pixels


## 1. Settings to edit

The next cell holds everything you normally need to change. It is set up for the example data.

### Using your own data: checklist

1. **Images.** Two folders per movie, each with **one 2-D, single-channel TIFF per frame**, named so that sorting
   by file name gives time order:
   - the **Ca²⁺ movie** (`calcium`; here GCaMP) and
   - the **membrane movie** (`membrane`), or any channel in which the wound is a dark hole.

   The two movies must show the same field of view, pixel for pixel, with the same frames. Put both folders in
   `data/wound/` (or point `WOUND_DATA_DIR` to where they are). Set `CALCIUM_GLOB` and `MEMBRANE_GLOB` to match
   the file names (e.g. `"*.tif"`). A multi-frame TIFF stack must first be saved as one file per frame (e.g. in
   Fiji: *File → Save As → Image Sequence*). If you have no membrane movie, set `membrane` to the Ca²⁺ folder: it
   is only used for the check in figure 0, but the wound must then be tracked in the Ca²⁺ movie.
2. **Wound tracking.** Track the wound with notebook 1 (set `MEMBRANE_FOLDER` there to your membrane folder).
   This notebook reads `wound_tracking/<membrane folder>/<TRACKING_CSV>`. Your own tracking also works: a CSV with
   the columns `frame, cy, cx, radius` (pixels; `cy` = row, `cx` = column; one row per frame, NaN where unknown).
3. **`DATASET`**: `name` (output folder), `condition` (label), the two folder names, and
   - `wound_frame`: the frame in which the wound is made. Times are in seconds after the start of this frame. Use
     the first frame in which the membrane inside the wound turns dark; figure 0 shows this, so you can check it
     after a first run.
   - `angles`: the part of the circle to measure, `(lo, hi)` in degrees, `0°` = 3 o'clock, increasing clockwise
     (`90°` = 6 o'clock, `180°` = 9 o'clock, `270°` = 12 o'clock); `(0, 360)` = the full circle, as used for every
     dataset of the paper. Use a smaller range only to keep out another injury; figure 2 shows the full circle.
4. **`PIXEL_SIZE_UM` and `FRAME_INTERVAL_S`**: µm per pixel and seconds per frame of your movie. All reported
   distances, times and speeds are converted with these.
5. **If your pixel size is not 1 µm:** the rings (section 2) are defined in pixels. To keep the paper's
   25-µm rings out to 200 µm, set `RING_STEP = round(25 / PIXEL_SIZE_UM)`, `RING_MAX_OFFSET = 8 * RING_STEP` and
   `FIT_MIN_OFFSET = RING_STEP`, and scale `MIN_PIXELS` with the ring area if your images are much finer or
   coarser.
6. **Run all cells.** Check figure 0 (the wound circle sits on the wound; the membrane darkens at `wound_frame`)
   figure 2 (full-circle maps) and figure 5 (sheet pixels), then read the verdict (section 9).

The method settings (section 2) are those of the paper; keep them to compare your data with the paper's.

In [ ]:
# =============================================================================
# SETTINGS TO EDIT (set up for the example data, ctrl_0820, with the settings used in the paper)
# =============================================================================

# ---- where the data are (see README.md; environment variables override these defaults) ----
REPO_ROOT = Path.cwd().parent   # this notebook runs from <repository>/notebooks
DATA_ROOT = Path(os.environ.get("WOUND_DATA_DIR", REPO_ROOT / "data" / "wound"))   # calcium + membrane folders
TRACKING_ROOT = Path(os.environ.get("WOUND_TRACKING_DIR", REPO_ROOT / "wound_tracking"))   # wound tracking
OUT_ROOT = Path(os.environ.get("RESULTS_DIR", REPO_ROOT / "results")) / "wound_wave_analysis"

# ---- the dataset ----
DATASET = dict(name="ctrl_0820",                  # name of the output folder
               condition="ctrl",                  # label in tables and figures
               calcium="ctrl_0820_calcium",       # folder in DATA_ROOT: Ca2+ movie, one TIFF per frame
               membrane="ctrl_0820_membrane",     # folder in DATA_ROOT: membrane movie, one TIFF per frame
               wound_frame=20,                    # frame in which the wound is made (time 0)
               angles=(0, 360))                   # measured part of the circle, degrees (0 = 3 o'clock, clockwise);
                                                  # (0, 360) = full circle
CALCIUM_GLOB = "AVG_Stack*.tif"                   # which files in the calcium folder are the frames
MEMBRANE_GLOB = "*.tif"                           # which files in the membrane folder are the frames
TRACKING_CSV = "hole_tracking_results_new.csv"    # in TRACKING_ROOT/<membrane folder>/, written by notebook 1

# ---- acquisition: set these for your microscope ----
PIXEL_SIZE_UM = 1.0       # micrometres per pixel
FRAME_INTERVAL_S = 7.0    # seconds per frame


## 2. Method settings (the values used in the paper)

**Measurement** (ring sizes in pixels; with 1 µm per pixel they equal µm)
- `RING_STEP` = 25, `RING_MAX_OFFSET` = 200: rings at 0, 25, 50, ..., 200 from the wound edge, each 25 wide
  (offset ± 12.5).
- `WEDGE_DEG` = 11.25°: 32 wedges around the full circle, all measured.
- `PRE_WOUND_FRAMES` = 2: the analysis starts 2 frames before the wound frame, so each trace has a pre-wound
  baseline.
- `SEED_RADIUS_TOL` = 1.3: guard against a wrong first tracked circle (section 3).
- `MIN_PIXELS` = 20: a sector is measured if it has ≥ 20 tissue pixels, and timed if it has ≥ 20 sheet pixels in
  every frame (otherwise it counts as not responding).
- `FIT_MIN_OFFSET` = 25: the ring on the wound edge (offset 0) is never used for timing (it contains the damaged
  edge itself).

**Sheet pixels** (`SHEET`, from `ca_wave/sheet_wave.py`; section 4 explains each)
- `SMOOTH_SIGMA_PX` = 2, `TISSUE_K` = 3: smoothing of the frames and the tissue mask.
- `ACTIVE_RISE` = 0.3, `ACTIVE_NOISE_K` = 4, `RISE_WINDOW` = 3: a pixel is active when its `(I - min)/min` rose by
  ≥ 0.3 and ≥ 4 × its noise above its lowest value of the last 3 frames.
- `RECRUIT_WINDOW` = 3, `ONSET_ONLY` = True, `LATE_TOL` = 1: how the sheet grows and which pixels are timed.
- `SOURCE_WINDOW_S` = 35: the band 0-12.5 µm outside the wound edge is the sheet's source for 35 s after the wound
  (as the 35-s uncaging light); after that the sheet grows only from its own front.

**Wave decision**
- `MIN_PEAK` = 0.2 and `NOISE_K` = 4: a sector responds if its peak `(I - min)/min` is at least 0.2 **and** at
  least 4 × its frame-to-frame noise **and** its trace starts below half of its peak.
- `MIN_FRAC_RESPONDING` = 0.5: a ring is reached when at least half of the measured wedges respond in it.
- `MIN_WAVE_RINGS` = 3: an outward wave must reach at least 3 consecutive rings from the first ring (75 µm).
- `INITIAL_WINDOW_S` = 21, `MIN_BEYOND_RINGS` = 2: the immediate response is the run of rings reached within 21 s
  of the wound; an outward wave must reach at least 2 rings (50 µm) beyond it.
- `N_PERMUTATIONS` = 5000, `ALPHA` = 0.05: the outward-order permutation test; `RANDOM_SEED` makes it repeatable.

**Plotting:** `NORM_VMAX` = 1.0, the same colour scale for every heatmap; `MONTAGE_AFTER_WOUND`, the frames
(relative to the wound frame) shown in figure 1.

The cell also checks the settings and the data (folders, number of frames, tracking file) and prints the scale.

In [ ]:
# =============================================================================
# METHOD SETTINGS: the values used in the paper (section 2 explains each one)
# Ring sizes and offsets are in PIXELS; with 1 um per pixel they equal micrometres.
# =============================================================================
NORM_LABEL = "(I-min)/min"

# ---- measurement ----
PRE_WOUND_FRAMES = 2         # each trace starts this many frames before the wound frame
SEED_RADIUS_TOL = 1.3        # first tracked circle off by > 30 % from the next frame's -> use the next frame's circle
RING_STEP = 25               # ring k covers hole_radius(t) + k*RING_STEP +- RING_STEP/2 (px)
RING_MAX_OFFSET = 200
WEDGE_DEG = 11.25
MIN_PIXELS = 20              # a sector needs >= this many tissue pixels to be measured, and >= this many
                             # sheet pixels (in every frame) to be timed
FIT_MIN_OFFSET = 25          # the ring on the wound edge (offset 0) is never used for timing

# ---- sheet wave (shared with the uncaging notebook: ca_wave/sheet_wave.py, values used in the paper) ----
SHEET = dict(SHEET_SETTINGS)  # SMOOTH_SIGMA_PX, TISSUE_K, ACTIVE_RISE, ACTIVE_NOISE_K, RISE_WINDOW, OPEN_ITER, LATE_TOL,
                              # RECRUIT_WINDOW, ONSET_ONLY
SOURCE_WINDOW_S = 35.0       # the wound edge band is the sheet's source for this long after the wound (as the
                             # 35-s uncaging light); later the sheet grows only from its own front

# ---- wave decision (same as the uncaging notebook) ----
MIN_PEAK = 0.2               # a sector responds if its peak (I-min)/min >= this ...
NOISE_K = 4.0                # ... and >= NOISE_K x its own frame-to-frame noise,
                             # ... and its trace starts below half of its peak (it rises after the start)
MIN_FRAC_RESPONDING = 0.5    # a ring is "reached" when >= this fraction of measured wedges respond
MIN_WAVE_RINGS = 3           # an outward wave must reach this many consecutive rings from 25 um (3 -> 75 um)
INITIAL_WINDOW_S = 21.0      # the immediate response = the rings reached within this time of the event (3 frames)
MIN_BEYOND_RINGS = 2         # an outward wave must reach this many rings beyond the immediate response (50 um)
N_PERMUTATIONS = 5000        # ring-label permutations for the outward-order test
ALPHA = 0.05
RANDOM_SEED = 0

# ---- plotting ----
NORM_VMAX = 1.0              # same colour scale for every heatmap of every dataset
MONTAGE_AFTER_WOUND = [-1, 0, 1, 2, 3, 5, 8, 12, 20]

OFFSETS = np.arange(0, RING_MAX_OFFSET + RING_STEP, RING_STEP)   # 0, 25, ..., 200
N_WEDGES = int(round(360 / WEDGE_DEG))
WEDGE_LO = np.arange(N_WEDGES) * WEDGE_DEG
WEDGE_LABELS = [f"{lo:g}-{lo + WEDGE_DEG:g}" for lo in WEDGE_LO]

# ---- checks ----
OUT_ROOT.mkdir(parents=True, exist_ok=True)
for key in ("calcium", "membrane"):
    if not (DATA_ROOT / DATASET[key]).exists():
        raise FileNotFoundError(f"{key} folder not found: {DATA_ROOT / DATASET[key]}")
if not (TRACKING_ROOT / DATASET["membrane"] / TRACKING_CSV).exists():
    raise FileNotFoundError(f"Wound tracking not found: {TRACKING_ROOT / DATASET['membrane'] / TRACKING_CSV}. "
                            "Run notebook 1 first (or set TRACKING_CSV).")
n_ca = len(list((DATA_ROOT / DATASET["calcium"]).glob(CALCIUM_GLOB)))
n_mem = len(list((DATA_ROOT / DATASET["membrane"]).glob(MEMBRANE_GLOB)))
if n_ca == 0:
    raise FileNotFoundError(f"No {CALCIUM_GLOB} files in {DATA_ROOT / DATASET['calcium']} (check CALCIUM_GLOB)")
if n_mem < n_ca:
    print(f"WARNING: {n_ca} calcium frames but only {n_mem} membrane frames; the two movies should match frame for frame")
if not 0 <= DATASET["wound_frame"] < n_ca:
    raise ValueError(f"wound_frame = {DATASET['wound_frame']}, but the calcium movie has frames 0-{n_ca - 1}")
print(f"Dataset:   {DATASET['name']} ({DATASET['condition']}): {n_ca} calcium frames, {n_mem} membrane frames")
print(f"Scale:     {PIXEL_SIZE_UM} um per pixel, {FRAME_INTERVAL_S} s per frame; rings {RING_STEP} px apart, "
      f"out to {RING_MAX_OFFSET} px ({RING_MAX_OFFSET * PIXEL_SIZE_UM:g} um)")
print(f"Tracking:  {TRACKING_ROOT / DATASET['membrane'] / TRACKING_CSV}")
print(f"Results:   {OUT_ROOT / DATASET['name']}")


## 3. Step 1: the wound circle in every frame

The tracking file has one row per membrane frame (`frame, cy, cx, radius`; NaN before tracking starts). The Ca²⁺
and membrane movies are pixel-aligned and their frames match one to one, so the membrane circle of a frame is used
for the Ca²⁺ image of the same frame.

The circle used in each Ca²⁺ frame (`hole_circles`):
- From the first tracked frame at or after `wound_frame` on: that frame's tracked circle. If a frame has no
  tracked circle, the previous one is kept.
- In the frames before that (the pre-wound baseline, and any frame between the wound frame and the first tracked
  frame): the first tracked circle.
- If the first tracked circle differs in radius from the next tracked frame by more than `SEED_RADIUS_TOL`
  (30 %), the next frame's circle is used instead (a guard against a wrong seed; it only moves the rings in the
  baseline frames).

So the rings move inward with the wound edge as the wound closes: ring *k* is always *k* × `RING_STEP` from the
current edge.

`hole_darkness` measures the membrane intensity inside the hole relative to just outside it. It drops when the
wound is made, and figure 0 plots it to check `wound_frame`.

In [ ]:
def load_stack(folder, pattern):
    paths = sorted((DATA_ROOT / folder).glob(pattern))
    if not paths:
        raise FileNotFoundError(f"No {pattern} files in {DATA_ROOT / folder}")
    return np.stack([tifffile.imread(p) for p in paths]).astype(float), paths


def load_hole_tracking(membrane_folder, n_frames, csv_name=TRACKING_CSV):
    # cy, cx, radius per frame from the wound tracking (notebook 1)
    d = pd.read_csv(TRACKING_ROOT / membrane_folder / csv_name)
    cy, cx, r = (np.full(n_frames, np.nan) for _ in range(3))
    for row in d.itertuples():
        f = int(row.frame)
        if f < n_frames:
            cy[f], cx[f], r[f] = row.cy, row.cx, row.radius
    return cy, cx, r


def hole_circles(cy, cx, r, wound_frame):
    # (cx, cy, r) used in every frame; returns circles[n_frames, 3] and the first tracked frame used
    n = len(cy)
    valid = np.isfinite(cy) & np.isfinite(cx) & np.isfinite(r)
    after = np.nonzero(valid & (np.arange(n) >= wound_frame))[0]
    if after.size == 0:
        raise ValueError("no tracked hole at or after the wound frame")
    ref = int(after[0])
    if after.size > 1:   # seed frame much larger/smaller than the next tracked frame -> start from the next one
        ratio = r[after[0]] / r[after[1]]
        if max(ratio, 1 / ratio) > SEED_RADIUS_TOL:
            print(f"    note: first tracked circle (frame {ref}, r={r[ref]:.0f}px) differs from frame "
                  f"{after[1]} (r={r[after[1]]:.0f}px) by >{SEED_RADIUS_TOL - 1:.0%}; frame {after[1]}'s circle is "
                  f"used for frames <= {ref}")
            ref = int(after[1])
    circles = np.empty((n, 3))
    last = (cx[ref], cy[ref], r[ref])
    for f in range(n):
        if f >= ref and valid[f]:
            last = (cx[f], cy[f], r[f])
        circles[f] = last if f >= ref else (cx[ref], cy[ref], r[ref])
    return circles, ref


def hole_darkness(mem, circles, ref):
    # membrane intensity inside the hole / just outside it, with the circle a few frames after tracking starts;
    # it drops when the wound is made (used to check wound_frame)
    cx, cy, r = circles[min(ref + 5, len(circles) - 1)]
    yy, xx = np.mgrid[:mem.shape[1], :mem.shape[2]]
    d = np.hypot(xx - cx, yy - cy)
    inside, outside = d < 0.6 * r, (d > r + 15) & (d < r + 45)
    return mem[:, inside].mean(1) / mem[:, outside].mean(1)

## 4. Steps 2-6: measurement and wave test

**Sheet pixels** (`ca_wave/sheet_wave.py`, the same code and settings for the wound and the uncaging movies).
The movies show two kinds of Ca²⁺ signal: the wave, one connected sheet of responding cells that grows out from
the source (here the band 0-12.5 µm outside the wound edge, for `SOURCE_WINDOW_S` = 35 s after the
wound; the hole itself is not tissue), and single cells that flash on their own before, during and long after the wave (they flash just as
much under suramin, where there is no wave, and before the wound is made). A sector average mixes the two, so a
flash can set a sector's arrival. Each sector is therefore measured on its sheet pixels only:
1. every frame is lightly smoothed (`SMOOTH_SIGMA_PX` = 2 px) and each pixel's trace is normalized as
   `(I - min)/min`; tissue pixels are those whose minimum over time is clearly above the dark background
   (`TISSUE_K`);
2. a pixel is **active** in a frame when its `(I - min)/min` has risen by at least 0.3 (`ACTIVE_RISE`) and 4 × its
   own frame-to-frame noise (`ACTIVE_NOISE_K`) above its lowest value of the last 3 frames (`RISE_WINDOW`); a
   cell that is bright and slowly dimming is not active;
3. the **sheet** starts at the source and grows: in every frame, each region of pixels that *start* to be active
   in that frame (`ONSET_ONLY`) joins the sheet if it touches the front, i.e. the source or pixels the sheet
   reached in the last 3 frames (`RECRUIT_WINDOW`). Pixels reached long ago do not recruit, so cells that flash
   next to the sheet minutes after it passed cannot chain a false wave outward;
4. a pixel that joined the sheet is **timed** unless it had already been active for more than one frame
   (`LATE_TOL`) before the sheet reached it (its own earlier flash). Active pixels the sheet never reaches are
   flashes and are left out.

Figure 5 shows, for every movie, the sheet pixels, when the sheet reached them, and the pixels left out.

**Step 2, sectors** (`sector_means_moving`). In every analysed frame, every pixel gets
- its distance from the wound edge, `d = distance from the wound centre - wound radius`, and
- its angle around the wound centre.

It belongs to wedge `floor(angle / 11.25°)` and to ring `floor((d + 12.5) / 25)`: ring 0 covers -12.5 to 12.5,
ring 1 covers 12.5 to 37.5 (centre 25), and so on up to 200 (pixels; µm at 1 µm per pixel). The sector's value in
that frame is the mean Ca²⁺ intensity of its sheet pixels; the ring on the wound edge (offset 0), which is never
used for timing and moves into the closing hole, is measured on all its pixels. A sector with fewer than `MIN_PIXELS` tissue pixels in
any frame (near the image border, past the fin edge) is not measured; a measured sector with fewer than
`MIN_PIXELS` sheet pixels counts as not responding. All 32 wedges are measured (`angles = (0, 360)`).

**Step 3, normalization** (`normalize`). Each sector's trace `I(t)` over the analysed frames (from
`wound_frame - 2` to the end of the movie) becomes `(I(t) - min I) / min I`, the rise above that sector's own
lowest value. This scales every sector by its own baseline, so bright and dim cells are comparable.

**Step 4, response and arrival time.** For each sector:
- **noise** (`frame_noise`): the robust SD of the frame-to-frame changes of its normalized trace
  (1.4826 × median absolute deviation / √2);
- **peak**: the maximum of its normalized trace;
- it **responds** if peak ≥ 0.2 (`MIN_PEAK`), peak ≥ 4 × noise (`NOISE_K`), and its first value is below half of
  its peak. The last rule rejects traces that start high and dim over the movie (bleaching, or a cell that is
  already bright): with `(I - min)/min` their "peak" is at the start, which is not a response to the wound;
- **arrival time** (`half_max_arrival`): the first time the trace reaches half of its peak, interpolated linearly
  between the two frames around the crossing, in frames after the wound frame (× `FRAME_INTERVAL_S` for seconds).

**Step 5, reach and outward-order test** (rings from `FIT_MIN_OFFSET` to `RING_MAX_OFFSET` of the measured
wedges).
- **Reach.** A ring is reached when ≥ 50 % of the measured wedges respond in it (`MIN_FRAC_RESPONDING`). The
  reach is the last ring of the unbroken run of reached rings that starts at the first ring (25 µm).
- **Outward order.** Kendall τ between distance and arrival time over all responding sectors: τ > 0 means arrival
  is later farther from the wound. Its p value comes from 5000 permutations of the ring distances (the same
  shuffle for every wedge): it is the fraction of shuffles whose τ is at least as large as the real one.
- **Immediate response.** The same unbroken run, counting only arrivals within 21 s of the wound
  (`INITIAL_WINDOW_S`): the region that responds at once.
- **Outward wave** = reach ≥ 3 rings **and** reach ≥ the immediate response + 2 rings (`MIN_BEYOND_RINGS`)
  **and** τ > 0 **and** p < 0.05. Activity that comes in from elsewhere (from
  the image edge, for example) arrives *earlier* far from the wound and fails this test.

**Step 6, speed** (only for an outward wave). For each reached ring, the median arrival time over its responding
wedges. A Theil-Sen line (the median of all pairwise slopes, robust to outliers) is fitted to median arrival time
(s) against distance (µm) over the reached rings; **speed = 1 / slope** (µm/s), and the 95 % interval of the slope
is converted the same way. If the wave crosses the rings within one or two frames, the slope is close to 0 and the
upper limit of the speed is not constrained (reported as "inf").

Each wedge also gets its own reach and speed (`per_wedge_summary.csv`), for reference.

In [ ]:
def sector_means_moving(stack, circles, frames, pix=None, tissue=None):
    # Mean intensity of every (wedge, ring) sector, rings measured from each frame's own hole edge, over the pixels
    # in `pix` (the sheet pixels; all pixels if None). Returns F[wedge, ring, k] for k over `frames`, the smallest
    # number of pixels averaged per sector, and the smallest number of tissue pixels per sector (over the frames).
    _, h, w = stack.shape
    yy, xx = np.mgrid[:h, :w]
    n_bins = N_WEDGES * len(OFFSETS)
    F = np.empty((N_WEDGES, len(OFFSETS), len(frames)))
    n_px = np.empty((N_WEDGES, len(OFFSETS), len(frames)), dtype=int)
    n_tis = np.empty((N_WEDGES, len(OFFSETS), len(frames)), dtype=int)
    for k, f in enumerate(frames):
        cx, cy, r0 = circles[f]
        dist_from_edge = np.hypot(xx - cx, yy - cy) - r0
        angle = np.degrees(np.arctan2(yy - cy, xx - cx)) % 360
        wedge_idx = np.floor(angle / WEDGE_DEG).astype(int) % N_WEDGES
        ring_idx = np.floor((dist_from_edge + RING_STEP / 2) / RING_STEP).astype(int)
        in_rings = (ring_idx >= 0) & (ring_idx < len(OFFSETS))
        lab_all = wedge_idx * len(OFFSETS) + ring_idx
        used = in_rings & ((pix if pix is not None else True) | (ring_idx == 0))   # the edge ring: all pixels
        label = lab_all[used]
        cnt = np.bincount(label, minlength=n_bins)
        sums = np.bincount(label, weights=stack[f][used], minlength=n_bins)
        F[..., k] = (sums / np.maximum(cnt, 1)).reshape(N_WEDGES, len(OFFSETS))
        n_px[..., k] = cnt.reshape(N_WEDGES, len(OFFSETS))
        t_used = in_rings & ((tissue if tissue is not None else True) | (ring_idx == 0))
        n_tis[..., k] = np.bincount(lab_all[t_used], minlength=n_bins).reshape(N_WEDGES, len(OFFSETS))
    min_px = n_px.min(axis=-1)
    F[min_px < MIN_PIXELS] = np.nan
    return F, min_px, n_tis.min(axis=-1)


def normalize(F):
    # (I - min)/min per sector; returns (normalized trace, min over time)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        fmin = np.nanmin(F, axis=-1)
    return (F - fmin[..., None]) / fmin[..., None], fmin


def frame_noise(norm):
    # robust SD of frame-to-frame changes (per sector)
    d = np.diff(norm, axis=-1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        mad = np.nanmedian(np.abs(d - np.nanmedian(d, axis=-1, keepdims=True)), axis=-1)
    return 1.4826 * mad / np.sqrt(2)


def half_max_arrival(norm):
    # peak and half-max arrival index (linear interpolation between frames) along the last axis
    filled = np.nan_to_num(norm, nan=-np.inf)
    peak = filled.max(axis=-1)
    thr = 0.5 * peak
    i = np.argmax(filled >= thr[..., None], axis=-1)
    v1 = np.take_along_axis(filled, i[..., None], -1)[..., 0]
    v0 = np.take_along_axis(filled, np.maximum(i - 1, 0)[..., None], -1)[..., 0]
    rising = (i > 0) & (v1 > v0)
    with np.errstate(invalid="ignore"):
        frac = np.divide(thr - v0, v1 - v0, out=np.zeros_like(thr), where=rising)
    arrival = np.where(i > 0, i - 1 + frac, 0.0)
    return np.where(np.isfinite(peak), peak, np.nan), arrival


def wedges_in_range(lo, hi):
    hi_u = hi + 360 if hi <= lo else hi
    centre = WEDGE_LO + WEDGE_DEG / 2
    centre = np.where(centre < lo, centre + 360, centre)
    return (centre > lo) & (centre < hi_u)


def leading_run(mask):
    return int(np.argmin(np.append(mask, False)))


def analyze_dataset(cfg, rng):
    ca, ca_paths = load_stack(cfg["calcium"], CALCIUM_GLOB)
    mem, mem_paths = load_stack(cfg["membrane"], MEMBRANE_GLOB)
    n = ca.shape[0]
    t0 = cfg["wound_frame"]
    csv_name = TRACKING_CSV
    cy, cx, r = load_hole_tracking(cfg["membrane"], max(n, mem.shape[0]), csv_name)
    circles, ref = hole_circles(cy, cx, r, t0)
    start = max(t0 - PRE_WOUND_FRAMES, 0)
    frames = np.arange(start, n)

    # the sheet wave: pixels reached by the sheet that grows from the wound edge (ca_wave/sheet_wave.py). The source
    # is the band 0-12.5 px outside the hole edge for SOURCE_WINDOW_S after the wound; the hole is not tissue.
    yy, xx = np.mgrid[:ca.shape[1], :ca.shape[2]]
    c_ref = circles[ref]
    hole = np.hypot(xx - c_ref[0], yy - c_ref[1]) < c_ref[2]

    source_frames = SOURCE_WINDOW_S / FRAME_INTERVAL_S

    def seed(k):                      # k indexes `frames`; the edge band is the source for SOURCE_WINDOW_S
        f = frames[k]
        if not t0 <= f < t0 + source_frames:
            return None
        d_edge = np.hypot(xx - circles[f][0], yy - circles[f][1]) - circles[f][2]
        return (d_edge >= 0) & (d_edge <= RING_STEP / 2)

    sheet = sheet_pixels(ca[frames], seed=seed, exclude=hole, settings=SHEET)
    F, n_px, n_tissue = sector_means_moving(ca, circles, frames, pix=sheet["timed"], tissue=sheet["tissue"])
    valid = n_tissue >= MIN_PIXELS           # sectors with enough tissue are measured (timed or not)
    norm, Fmin = normalize(F)
    noise = frame_noise(norm)
    peak, arr_idx = half_max_arrival(norm)
    arrival = arr_idx + start - t0                       # frames after the wound frame
    threshold = np.maximum(MIN_PEAK, NOISE_K * noise)
    with np.errstate(invalid="ignore"):
        starts_low = norm[..., 0] < 0.5 * peak           # must rise after the start
    responds = np.isfinite(peak) & (peak >= threshold) & starts_low

    in_range = wedges_in_range(*cfg["angles"])
    fit_rings = OFFSETS >= FIT_MIN_OFFSET
    offs = OFFSETS[fit_rings]
    resp_m = responds[in_range][:, fit_rings]
    valid_m = valid[in_range][:, fit_rings]
    arr_m = arrival[in_range][:, fit_rings]

    frac_resp = resp_m.sum(0) / np.maximum(valid_m.sum(0), 1)
    n_reached = leading_run(frac_resp >= MIN_FRAC_RESPONDING)
    reach_um = float(offs[n_reached - 1] * PIXEL_SIZE_UM) if n_reached else 0.0
    median_arrival = np.array([np.median(arr_m[resp_m[:, k], k]) if resp_m[:, k].any() else np.nan
                               for k in range(len(offs))])

    w_idx, r_idx = np.nonzero(resp_m)
    y = arr_m[w_idx, r_idx]
    tau, p, null = np.nan, np.nan, np.array([])
    if len(np.unique(r_idx)) >= 3 and np.ptp(y) > 0:
        tau = kendalltau(offs[r_idx], y).statistic
        null = np.array([kendalltau(offs[rng.permutation(len(offs))][r_idx], y).statistic
                         for _ in range(N_PERMUTATIONS)])
        p = (1 + np.sum(null >= tau)) / (1 + N_PERMUTATIONS)

    # the immediate response: rings where >= MIN_FRAC_RESPONDING of the wedges respond within INITIAL_WINDOW_S
    early_m = resp_m & (arr_m <= INITIAL_WINDOW_S / FRAME_INTERVAL_S)
    n_initial = leading_run(early_m.sum(0) / np.maximum(valid_m.sum(0), 1) >= MIN_FRAC_RESPONDING)
    initial_reach_um = float(offs[n_initial - 1] * PIXEL_SIZE_UM) if n_initial else 0.0

    is_wave = bool(n_reached >= MIN_WAVE_RINGS and n_reached >= n_initial + MIN_BEYOND_RINGS
                   and np.isfinite(tau) and tau > 0 and p < ALPHA)
    if is_wave:
        verdict = "outward wave"
    elif n_reached < MIN_WAVE_RINGS:
        verdict = f"no outward wave: responses reach only {reach_um:.0f} um"
    elif n_reached < n_initial + MIN_BEYOND_RINGS:
        verdict = (f"no outward wave: the response does not spread beyond the immediate response "
                   f"({initial_reach_um:.0f} um within {INITIAL_WINDOW_S:.0f} s; reach {reach_um:.0f} um)")
    else:
        verdict = f"no outward wave: arrival not later with distance (tau={tau:.2f}, p={p:.3f})"

    speed = speed_lo = speed_hi = np.nan
    fit = None
    if is_wave:
        x_um = offs[:n_reached] * PIXEL_SIZE_UM
        t_s = median_arrival[:n_reached] * FRAME_INTERVAL_S
        slope, intercept, lo_slope, hi_slope = theilslopes(t_s, x_um)
        fit = (slope, intercept, x_um)
        if slope > 0:
            speed = 1.0 / slope
            speed_lo = 1.0 / hi_slope if hi_slope > 0 else np.nan
            speed_hi = 1.0 / lo_slope if lo_slope > 0 else np.inf
        else:
            is_wave, verdict = False, "no outward wave: median arrival does not increase with distance"

    wedge_rows = []
    for j, w in enumerate(np.nonzero(in_range)[0]):
        rr = resp_m[j]
        nc = leading_run(rr)
        tau_w = kendalltau(offs[rr], arr_m[j, rr]).statistic if rr.sum() >= 3 and np.ptp(arr_m[j, rr]) > 0 else np.nan
        speed_w = np.nan
        if nc >= MIN_WAVE_RINGS:
            s_w = theilslopes(arr_m[j, :nc] * FRAME_INTERVAL_S, offs[:nc] * PIXEL_SIZE_UM)[0]
            speed_w = 1.0 / s_w if s_w > 0 else np.nan
        wedge_rows.append({"sub_wedge": WEDGE_LABELS[w], "n_responding_rings": int(rr.sum()),
                           "consecutive_rings_from_25px": nc,
                           "reach_um": float(offs[nc - 1] * PIXEL_SIZE_UM) if nc else 0.0,
                           "kendall_tau": tau_w, "speed_um_per_s": speed_w})

    return dict(cfg=cfg, tracking_csv=csv_name, ca=ca, mem=mem, n_ca_files=len(ca_paths), circles=circles, ref=ref, tracked_r=r[:mem.shape[0]],
                t0=t0, start=start, frames=frames, darkness=hole_darkness(mem, circles, ref),
                sheet=sheet, n_tissue=n_tissue, valid=valid, F=F, Fmin=Fmin, n_px=n_px, norm=norm, noise=noise, peak=peak, starts_low=starts_low,
                arrival=arrival, threshold=threshold, responds=responds, in_range=in_range, offs=offs,
                resp_m=resp_m, arr_m=arr_m, frac_resp=frac_resp, n_reached=n_reached, reach_um=reach_um,
                n_initial=n_initial, initial_reach_um=initial_reach_um, median_arrival=median_arrival, tau=tau, p=p, null=null, is_wave=is_wave, verdict=verdict,
                speed=speed, speed_lo=speed_lo, speed_hi=speed_hi, fit=fit, wedge_df=pd.DataFrame(wedge_rows))

## 5. Saving and plotting helpers

Tables (in `results/wound_wave_analysis/<dataset>/`):
- `hole_circle_used.csv`: the wound circle used in every analysed frame.
- `sector_intensity_over_time.csv`: every wedge × ring × frame of the full circle: mean intensity, minimum over
  time, normalized value (means over the sector's sheet pixels); `in_measured_range` marks the measured wedges.
- `sector_arrival_times.csv`: per sector, its tissue and sheet pixel counts, whether it is measured, its peak, noise, threshold, whether it starts below half of its peak,
  whether it responds, and its arrival time.
- `ring_summary.csv`: per ring, the fraction of wedges responding, the fraction of tissue pixels in the sheet, whether it is reached, and the median arrival.
- `per_wedge_summary.csv`: per wedge, its reach, Kendall τ and speed.

Figures (in `figures/`, as PNG and PDF):
- **0, wound tracking check:** radius over time, membrane darkness at the hole (drops at the wound frame), and the
  wound circle on the membrane image at the start and the end.
- **1, overview:** the Ca²⁺ image with the wound circle, rings and measured part of the circle, then `(I - min)/min` images at
  several times after the wound.
- **2, full-circle maps:** response and arrival time of every sector around the full circle, to see where activity
  comes from.
- **3, kymographs:** normalized intensity against time and distance for every measured wedge, with the arrival
  times.
- **4, arrival vs distance:** every responding sector's arrival time against distance, the median per ring, and
  the Theil-Sen fit with the speed.
- **5, sheet pixels:** the pixels the sheet reached and when, and the pixels left out (active before the sheet
  came, or flashes the sheet never reached).

In [ ]:
def save_figure(fig, folder, name, dpi=150):
    folder.mkdir(parents=True, exist_ok=True)
    for ext in ("png", "pdf"):
        fig.savefig(folder / f"{name}.{ext}", dpi=dpi, bbox_inches="tight")


def save_tables(res, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    frames, t0 = res["frames"], res["t0"]
    circ = res["circles"][frames]
    pd.DataFrame({"frame": frames, "time_after_wound_s": (frames - t0) * FRAME_INTERVAL_S,
                  "hole_cx": circ[:, 0], "hole_cy": circ[:, 1], "hole_radius_px": circ[:, 2],
                  "tracked_this_frame": frames >= res["ref"]}).to_csv(out_dir / "hole_circle_used.csv", index=False)

    w, r, k = np.meshgrid(np.arange(N_WEDGES), np.arange(len(OFFSETS)), np.arange(len(frames)), indexing="ij")
    pd.DataFrame({
        "sub_wedge": np.array(WEDGE_LABELS)[w.ravel()], "in_measured_range": res["in_range"][w.ravel()],
        "offset_px": OFFSETS[r.ravel()], "frame": frames[k.ravel()],
        "time_after_wound_s": (frames[k.ravel()] - t0) * FRAME_INTERVAL_S,
        "mean_intensity": res["F"].ravel(), "min_over_time": res["Fmin"][w.ravel(), r.ravel()],
        "normalized_intensity": res["norm"].ravel(),
    }).to_csv(out_dir / "sector_intensity_over_time.csv", index=False)

    w2, r2 = np.meshgrid(np.arange(N_WEDGES), np.arange(len(OFFSETS)), indexing="ij")
    pd.DataFrame({
        "sub_wedge": np.array(WEDGE_LABELS)[w2.ravel()], "in_measured_range": res["in_range"][w2.ravel()],
        "offset_px": OFFSETS[r2.ravel()], "min_n_tissue_pixels": res["n_tissue"].ravel(),
        "min_n_sheet_pixels": res["n_px"].ravel(), "measured": res["valid"].ravel(),
        "peak_normalized": res["peak"].ravel(), "noise_normalized": res["noise"].ravel(),
        "response_threshold": res["threshold"].ravel(), "starts_below_half_peak": res["starts_low"].ravel(),
        "responds": res["responds"].ravel(),
        "arrival_s_after_wound": np.where(res["responds"], res["arrival"] * FRAME_INTERVAL_S, np.nan).ravel(),
    }).to_csv(out_dir / "sector_arrival_times.csv", index=False)

    pd.DataFrame({
        "offset_px": res["offs"], "offset_um": res["offs"] * PIXEL_SIZE_UM,
        "fraction_wedges_responding": res["frac_resp"],
        "fraction_tissue_in_sheet": sheet_fraction(res),
        "reached": np.arange(len(res["offs"])) < res["n_reached"],
        "median_arrival_s_after_wound": res["median_arrival"] * FRAME_INTERVAL_S,
    }).to_csv(out_dir / "ring_summary.csv", index=False)
    res["wedge_df"].to_csv(out_dir / "per_wedge_summary.csv", index=False)


def sheet_fraction(res):
    # per ring (>= FIT_MIN_OFFSET, measured wedges): sheet pixels / tissue pixels (smallest counts over the frames)
    fit_rings = OFFSETS >= FIT_MIN_OFFSET
    n_s = res["n_px"][res["in_range"]][:, fit_rings].sum(0)
    n_t = res["n_tissue"][res["in_range"]][:, fit_rings].sum(0)
    return n_s / np.maximum(n_t, 1)


def plot_sheet_maps(res):
    # which pixels the sheet reached and when (after the wound), and which were left out
    cfg, sh = res["cfg"], res["sheet"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    t_s = np.where(sh["timed"], (sh["t_reach"] + res["start"] - res["t0"]) * FRAME_INTERVAL_S, np.nan)
    im = axes[0].imshow(t_s, cmap="viridis")
    fig.colorbar(im, ax=axes[0], shrink=0.8, label="time the sheet reached the pixel (s after the wound)")
    axes[0].set_title("sheet pixels (timed): when the sheet reached them", fontsize=10)
    cls = np.zeros(sh["tissue"].shape)
    cls[sh["tissue"]] = 1
    cls[sh["timed"]] = 2
    cls[sh["early"]] = 3
    cls[sh["flash_only"]] = 4
    colors = ["black", "0.35", "tab:green", "tab:orange", "tab:red"]
    axes[1].imshow(cls, cmap=ListedColormap(colors), vmin=-0.5, vmax=4.5, interpolation="nearest")
    labels = ["not tissue (or the hole)", "tissue, not reached", "sheet (timed)",
              "active before the sheet came (not timed)", "active, never reached by the sheet (flashes)"]
    axes[1].legend(handles=[Patch(color=cc, label=l) for cc, l in zip(colors, labels)], fontsize=7, loc="lower left")
    axes[1].set_title("pixel classes", fontsize=10)
    for ax in axes:
        draw_hole(ax, res["circles"][res["ref"]], cfg["angles"])
        ax.axis("off")
    fig.suptitle(f"{cfg['name']}: sheet wave (only green pixels are used for the sector traces)", y=1.0)
    fig.tight_layout()
    return fig


def draw_hole(ax, circle, angles=None, rings=False):
    cx, cy, r0 = circle
    ax.add_patch(Circle((cx, cy), r0, fill=False, edgecolor="lime", linewidth=1.3, linestyle="--"))
    if rings:
        for off in OFFSETS[1:]:
            ax.add_patch(Circle((cx, cy), r0 + off, fill=False, edgecolor="cyan", linewidth=0.5, alpha=0.6))
    if angles is not None:
        lo, hi = angles
        hi_u = hi + 360 if hi <= lo else hi
        outer = r0 + RING_MAX_OFFSET + RING_STEP / 2
        ax.add_patch(Wedge((cx, cy), outer, lo, hi_u, width=outer - r0, facecolor="none", edgecolor="orange",
                           linewidth=1.2, linestyle="--"))


def plot_tracking(res):
    # the kept wound tracking: radius over time, membrane darkness at the hole, and the circle on the images
    cfg, t0, ref = res["cfg"], res["t0"], res["ref"]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.4))
    n_mem = len(res["tracked_r"])
    ax = axes[0]
    ax.plot(np.arange(n_mem), res["tracked_r"], "o-", ms=3, color="0.3", label="tracked radius (notebook 1)")
    ax.plot(res["frames"], res["circles"][res["frames"], 2], "-", color="tab:green", lw=2, label="radius used here")
    ax.axvline(t0, color="tab:red", ls="--", label=f"wound frame {t0}")
    ax.axvline(res["start"], color="0.5", ls=":", label=f"analysis start {res['start']}")
    ax.set_xlabel("frame")
    ax.set_ylabel("hole radius (px)")
    ax.legend(fontsize=7)
    ax.set_title("wound radius")
    ax = axes[1]
    ax.plot(res["darkness"], "o-", ms=3, color="0.3")
    ax.axvline(t0, color="tab:red", ls="--")
    ax.set_xlabel("frame")
    ax.set_ylabel("membrane: inside hole / around hole")
    ax.set_title("hole darkness (drops when the wound is made)")
    mem = res["mem"]
    vmin, vmax = np.percentile(mem, [1, 99.5])
    for ax, f, ttl in [(axes[2], min(ref + 1, len(mem) - 1), "membrane, first tracked frame + 1"),
                       (axes[3], min(res["ca"].shape[0], len(mem)) - 1, "membrane, last analysed frame")]:
        ax.imshow(mem[f], cmap="gray", vmin=vmin, vmax=vmax)
        draw_hole(ax, res["circles"][f], cfg["angles"])
        ax.set_title(f"{ttl} (f{f})", fontsize=9)
        ax.axis("off")
    fig.suptitle(f"{cfg['name']} ({cfg['condition']}): wound tracking from {res['tracking_csv']}; "
                 f"dashed orange = measured wedges", y=1.02)
    fig.tight_layout()
    return fig


def plot_overview(res):
    cfg, ca, t0 = res["cfg"], res["ca"], res["t0"]
    frames = [t0 + d for d in MONTAGE_AFTER_WOUND if res["start"] <= t0 + d < ca.shape[0]]
    base = gaussian(ca[res["frames"]], sigma=(0, 1.5, 1.5), preserve_range=True).min(axis=0)
    fig, axes = plt.subplots(2, 5, figsize=(20, 8.4))
    axes = axes.ravel()
    f1 = min(t0 + 1, ca.shape[0] - 1)
    vmin, vmax = np.percentile(ca[f1], [1, 99.5])
    axes[0].imshow(ca[f1], cmap="gray", vmin=vmin, vmax=vmax)
    draw_hole(axes[0], res["circles"][f1], cfg["angles"], rings=True)
    axes[0].set_title(f"calcium frame {f1} + hole + rings + measured wedges", fontsize=9)
    for ax, f in zip(axes[1:], frames):
        ax.imshow((ca[f] - base) / np.maximum(base, 1e-6), cmap="magma", vmin=0, vmax=NORM_VMAX)
        draw_hole(ax, res["circles"][f], cfg["angles"])
        ax.set_title(f"{NORM_LABEL} frame {f} ({(f - t0) * FRAME_INTERVAL_S:+.0f} s from wound)", fontsize=9)
    for ax in axes:
        ax.axis("off")
    for ax in axes[1 + len(frames):]:
        ax.set_visible(False)
    fig.suptitle(f"{cfg['name']} ({cfg['condition']}): {NORM_LABEL}, fixed scale 0-{NORM_VMAX:g}", y=1.0)
    fig.tight_layout()
    return fig


def plot_angle_maps(res):
    # full 360 deg (display only): peak and arrival per sector, to check the measured wedges
    cfg = res["cfg"]
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
    ext = [0, 360, OFFSETS[0] - RING_STEP / 2, OFFSETS[-1] + RING_STEP / 2]
    im0 = axes[0].imshow(res["peak"].T, aspect="auto", origin="lower", extent=ext, cmap="magma", vmin=0, vmax=NORM_VMAX)
    fig.colorbar(im0, ax=axes[0], label=f"peak {NORM_LABEL}")
    arr_s = np.where(res["responds"], res["arrival"] * FRAME_INTERVAL_S, np.nan)
    im1 = axes[1].imshow(arr_s.T, aspect="auto", origin="lower", extent=ext, cmap="viridis")
    fig.colorbar(im1, ax=axes[1], label="arrival (s after wound), responding sectors only")
    lo, hi = cfg["angles"]
    spans = [(lo, hi)] if hi > lo else [(lo, 360), (0, hi)]
    for ax in axes:
        for a, b in spans:
            ax.axvspan(a, b, facecolor="none", edgecolor="orange", lw=2.5, ls="--")
        ax.axhline(FIT_MIN_OFFSET - RING_STEP / 2, color="white", lw=0.8, ls=":")
        ax.set_xlabel("angle (deg, 0 = 3 o'clock, clockwise)")
        ax.set_ylabel("distance from wound edge (px)")
        ax.set_xticks(np.arange(0, 361, 45))
    axes[0].set_title(f"peak {NORM_LABEL}, full circle (orange = measured wedges)")
    axes[1].set_title("arrival time: an outward wave is earliest at small distance")
    fig.suptitle(cfg["name"], y=1.02)
    fig.tight_layout()
    return fig


def plot_kymographs(res):
    cfg = res["cfg"]
    idx = np.nonzero(res["in_range"])[0]
    rel = res["frames"] - res["t0"]
    ncols = 4
    nrows = int(np.ceil(len(idx) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 3.3 * nrows), squeeze=False, layout="constrained")
    axes = axes.ravel()
    for ax, w in zip(axes, idx):
        im = ax.imshow(res["norm"][w].T, aspect="auto", origin="lower", cmap="magma", vmin=0, vmax=NORM_VMAX,
                       extent=[OFFSETS[0] - RING_STEP / 2, OFFSETS[-1] + RING_STEP / 2, rel[0] - 0.5, rel[-1] + 0.5])
        rsp = res["responds"][w] & (OFFSETS >= FIT_MIN_OFFSET)
        ax.plot(OFFSETS[rsp], res["arrival"][w][rsp], "o", color="white", mec="black", ms=5)
        ax.axhline(0, color="cyan", ls="--", lw=1)
        ax.axvline(FIT_MIN_OFFSET - RING_STEP / 2, color="white", lw=0.6, ls=":")
        ax.set_title(f"{WEDGE_LABELS[w]} deg  ({rsp.sum()} rings respond)", fontsize=9)
        ax.set_xlabel("offset from wound edge (px)")
        ax.set_ylabel("frames after wound")
    for ax in axes[len(idx):]:
        ax.axis("off")
    fig.colorbar(im, ax=axes[:len(idx)].tolist(), shrink=0.6, label=f"{NORM_LABEL} (same scale for all datasets)")
    fig.suptitle(f"{cfg['name']}: {NORM_LABEL} per wedge; white dots = half-max arrival of responding sectors; "
                 f"cyan dashed = wound frame")
    return fig


def plot_arrival_vs_distance(res):
    cfg = res["cfg"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [2, 1]})
    ax = axes[0]
    offs_um = res["offs"] * PIXEL_SIZE_UM
    w_idx, r_idx = np.nonzero(res["resp_m"])
    jitter = np.random.default_rng(1).uniform(-3, 3, len(r_idx))
    ax.plot(offs_um[r_idx] + jitter, res["arr_m"][w_idx, r_idx] * FRAME_INTERVAL_S, "o", color="0.6", ms=4,
            alpha=0.6, label="responding sector (one per wedge)")
    ax.plot(offs_um, res["median_arrival"] * FRAME_INTERVAL_S, "s-", color="black", ms=7, label="median per ring")
    if res["n_reached"]:
        ax.axvspan(offs_um[0] - 12.5, offs_um[res["n_reached"] - 1] + 12.5, color="gold", alpha=0.15,
                   label=f"reached rings (>= {MIN_FRAC_RESPONDING:.0%} of wedges respond)")
    if res["is_wave"]:
        slope, intercept, x_um = res["fit"]
        ax.plot(x_um, slope * x_um + intercept, "-", color="tab:red", lw=2.5,
                label=f"Theil-Sen: {res['speed']:.2f} um/s (95% CI {res['speed_lo']:.2f}-{res['speed_hi']:.2f})")
    ax.set_xlabel("distance from wound edge (um)")
    ax.set_ylabel("arrival time (s after wound)")
    ax.set_xlim(0, OFFSETS[-1] * PIXEL_SIZE_UM + 15)
    ax.legend(fontsize=8, loc="upper left")
    ax.set_title(f"{cfg['name']}: {res['verdict']}", fontsize=10)
    ax = axes[1]
    if res["null"].size:
        ax.hist(res["null"], bins=40, color="0.75")
        ax.axvline(res["tau"], color="tab:red", lw=2, label=f"observed tau = {res['tau']:.2f}")
        ax.set_title(f"outward-order test: p = {res['p']:.4f}", fontsize=10)
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "too few responding rings\nfor the order test", ha="center", va="center",
                transform=ax.transAxes)
    ax.set_xlabel("Kendall tau (distance vs arrival)")
    ax.set_ylabel("ring-label permutations")
    fig.tight_layout()
    return fig

## 6. Run

Measures the dataset, saves the tables and figures, and prints the per-ring response fractions, the median arrival
per ring, the immediate response, the reach, the outward-order test and the verdict.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
cfg = DATASET
res = analyze_dataset(cfg, rng)
out_dir = OUT_ROOT / cfg["name"]
save_tables(res, out_dir)
print(f"=== {cfg['name']} ({cfg['condition']}): {res['ca'].shape[0]} calcium frames, wound frame {res['t0']}, "
      f"frames {res['frames'][0]}-{res['frames'][-1]} analysed, tracked circle from frame {res['ref']} "
      f"(r = {res['circles'][res['ref'], 2]:.0f} -> {res['circles'][res['frames'][-1], 2]:.0f} px)")
print(f"    fraction of wedges responding per ring (25..200 um): {np.round(res['frac_resp'], 2).tolist()}")
print(f"    median arrival per ring (s after the wound): {np.round(res['median_arrival'] * FRAME_INTERVAL_S, 1).tolist()}")
print(f"    immediate response {res['initial_reach_um']:.0f} um (within {INITIAL_WINDOW_S:.0f} s), reach = {res['reach_um']:.0f} um, tau = {res['tau']:.2f}, p = {res['p']:.4f}")
print(f"    -> {res['verdict']}" + (f", speed = {res['speed']:.2f} um/s "
                                    f"(95% CI {res['speed_lo']:.2f}-{res['speed_hi']:.2f})" if res["is_wave"] else ""))
for fig_fn, fig_name in [(plot_tracking, "0_wound_tracking_check"), (plot_overview, "1_overview_normalized_frames"),
                         (plot_angle_maps, "2_full_circle_maps"), (plot_kymographs, "3_kymographs_per_wedge"),
                         (plot_arrival_vs_distance, "4_arrival_vs_distance"), (plot_sheet_maps, "5_sheet_pixels")]:
    fig = fig_fn(res)
    save_figure(fig, out_dir / "figures", fig_name)
    plt.show()


## 7. Summary

One row with the verdict and the numbers, saved as `wave_summary.csv`.

In [ ]:
summary_df = pd.DataFrame([{
    "dataset": name, "condition": res["cfg"]["condition"], "calcium_folder": res["cfg"]["calcium"],
    "membrane_folder": res["cfg"]["membrane"], "tracking_csv": res["tracking_csv"], "n_calcium_frames": res["ca"].shape[0],
    "wound_frame": res["t0"], "first_frame_analysed": int(res["frames"][0]), "first_tracked_frame_used": res["ref"],
    "hole_radius_at_wound_px": res["circles"][res["ref"], 2], "hole_radius_last_px": res["circles"][res["frames"][-1], 2],
    "measured_angles": f"{res['cfg']['angles'][0]}-{res['cfg']['angles'][1]}",
    "initial_response_reach_um": res["initial_reach_um"], "response_reach_um": res["reach_um"], "kendall_tau": res["tau"], "order_test_p": res["p"],
    "outward_wave": res["is_wave"], "verdict": res["verdict"],
    "wave_reach_um": res["reach_um"] if res["is_wave"] else 0.0,
    "speed_um_per_s": res["speed"], "speed_ci_low": res["speed_lo"], "speed_ci_high": res["speed_hi"],
    "median_per_wedge_speed_um_per_s": res["wedge_df"]["speed_um_per_s"].median() if res["is_wave"] else np.nan,
} for name, res in {cfg['name']: res}.items()])
summary_df.to_csv(out_dir / "wave_summary.csv", index=False)
print(f"Saved: {out_dir / 'wave_summary.csv'}")
summary_df[["dataset", "condition", "wound_frame", "outward_wave", "verdict", "response_reach_um", "kendall_tau",
            "order_test_p", "speed_um_per_s", "speed_ci_low", "speed_ci_high"]].round(4).T

## 8. Result for the example (ctrl_0820)

- **Outward wave**, reaching 200 µm: at least half of the wedges respond in every ring from 25 to 200 µm. The
  immediate response (within 21 s of the wound) covers the 25-µm ring.
- Arrival is clearly later farther from the wound: Kendall τ = 0.56, p = 0.0002 (none of the 5000 shuffles
  reached it).
- **Speed 2.43 µm/s** (95 % CI 1.39-4.09), the Theil-Sen fit of the median arrival per ring.

## 9. How to read the results, and troubleshooting

**Verdicts**
- **"outward wave"**: responses in an unbroken run from 25 µm out to at least 75 µm, later arrival at larger
  distance (τ > 0), and p < 0.05 against the permutations. Only then is a speed reported.
- **"no outward wave: responses reach only X µm"**: cells next to the wound respond, but the response does not
  spread far enough.
- **"no outward wave: the response does not spread beyond the immediate response"**: a region around the wound
  responds at once (within 21 s), but the response does not travel on from there.
- **"no outward wave: arrival not later with distance"**: there is activity in the measured wedges, but it is
  not ordered outward (figure 2 shows where it comes from).

**Troubleshooting**

| Problem | What to change |
|---|---|
| `... folder not found` or `No ... files` | Folder names in `DATASET`, `CALCIUM_GLOB` / `MEMBRANE_GLOB`, or `WOUND_DATA_DIR`. |
| `Wound tracking not found` | Run notebook 1 for this dataset, or set `TRACKING_CSV`. |
| `no tracked hole at or after the wound frame` | The tracking starts after the last frame or is empty: check notebook 1, and `wound_frame`. |
| The membrane does not darken at `wound_frame` in figure 0 | Set `wound_frame` to the frame where it does, and run again. |
| The circle in figure 0 or 1 is not on the wound | Re-track the wound with notebook 1. |
| Another injury is in the field (figure 2) | Set `angles` to the part of the circle away from it. |
| The sheet misses a clear wave, or takes in flashing cells (figure 5) | Check the tissue mask and the sheet in figure 5; the sheet settings (`SHEET`) set what counts as active (changing them departs from the paper's method). |
| Very few sectors respond although the movie shows a clear wave | Check `PIXEL_SIZE_UM` and the ring sizes (section 1, item 5); for very dim or noisy movies, `MIN_PEAK` and `NOISE_K` set what counts as a response (changing them departs from the paper's method). |

## Appendix: settings used for the paper's datasets

All datasets: 1 µm per pixel, 7 s per frame, `angles = (0, 360)`, the method settings of section 2.

| Dataset | Condition | Calcium folder | Membrane folder | `wound_frame` | Wound tracking (`TRACKING_CSV`) |
|---|---|---|---|---|---|
| ctrl_0808 | ctrl | ctrl_0808_calcium | ctrl_0808_membrane | 1 | notebook 1, `"halfway"` (`hole_tracking_results_new.csv`) |
| ctrl_0820 | ctrl | ctrl_0820_calcium | ctrl_0820_membrane | 20 | notebook 1, `"halfway"` (`hole_tracking_results_new.csv`); the example |
| pom1-3 | POM1 | pom1-3_calcium | pom1-3_membrane | 2 | supplied (`hole_tracking_results.csv`) |
| Gd3_3 | Gd3+ | Gd3_3_calcium | Gd3_3_membrane | 1 | supplied (`hole_tracking_results.csv`) |
| Gd3-4 | Gd3+ | Gd3-4-calcium | Gd3-4-membrane | 1 | notebook 1, `"halfway"` (`hole_tracking_results_new.csv`) |
| CBX_5 | CBX | CBX_5_calcium | CBX_5_membrane | 20 | supplied (`hole_tracking_results.csv`) |
| CBX_6 | CBX | CBX_6_calcium | CBX_6_membrane | 20 | notebook 1, `"halfway"` (`hole_tracking_results_new.csv`) |
| CBX_7 | CBX | CBX_7_calcium | CBX_7_membrane | 20 | supplied (`hole_tracking_results.csv`) |
| hep_6 | heptanol | hep_6_calcium | hep_6_membrane | 20 | notebook 1, `"halfway"` (`hole_tracking_results_new.csv`) |
| hep_7 | heptanol | hep_7_calcium | hep_7_membrane | 20 | supplied (`hole_tracking_results.csv`) |
| heptanol_3 | heptanol | heptonal_3_calcium | heptonal_3_membrane | 20 | supplied (`hole_tracking_results.csv`) |
| heptanol_5 | heptanol | heptonal_5_calcium | heptanol_5_membrane | 20 | supplied (`hole_tracking_results.csv`) |

The notebook 1 settings of the `"halfway"` datasets are in the appendix of notebook 1. The wound dataset suramin_1
is excluded from the analysis (data-quality issue).